# 06d. Собираем Transformer-блок

Лекция: [Pre-norm Transformer-блок](../../notes/interview-prep/08j-transformer-block.md).

Сегодня не переписываем attention. Он перенесён из твоего урока 05c; старый комментарий TODO заменён пояснением. Задача A — две строки forward. Сначала только A.

Ментор не запускает ноутбук и проверки и не коммитит без отдельной просьбы.

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(42)


## Готовый компонент: твой attention из 05c

Эту ячейку достаточно выполнить. Повторно выводить reshape и transpose сейчас не нужно.

In [2]:
class VectorizedCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def project_heads(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,D)')
        batch_size, length, _ = x.shape
        # Готовая реализация из урока 05c: проекция и разделение голов.
        q, k, v = [
            projection(x)
            .reshape(batch_size, length, self.num_heads, self.head_dim)
            .transpose(1, 2)
            for projection in (
                self.query_projection,
                self.key_projection,
                self.value_projection,
            )
        ]

        return q, k, v

    def forward(self, x):
        batch_size, length, _ = x.shape
        q, k, v = self.project_heads(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        combined = context.transpose(1, 2).reshape(batch_size, length, self.d_model)
        return self.output_projection(combined)

## A. Соединить две residual-ветви

Все модули уже созданы. Заполни две строки в forward:

1. Прибавь к x результат attention от norm_1(x).
2. Прибавь к обновлённому x результат mlp от norm_2(x).

Используй обычное присваивание `x = x + ...`, не `+=`. Подсказка: `self.attention(self.norm_1(x))` — результат первой ветви.

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.norm_1 = nn.LayerNorm(d_model)
        self.attention = VectorizedCausalAttention(d_model, num_heads)
        self.norm_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        x = x + self.attention(self.norm_1(x))
        x = x + self.mlp(self.norm_2(x))
        return x


In [4]:
block = TransformerBlock(d_model=8, num_heads=2, d_ff=32)
x = torch.randn(2, 4, 8)
y = block(x)

print('Вход:', x.shape)
print('Выход:', y.shape)
print('Количество параметров:', sum(p.numel() for p in block.parameters()))
# Ожидаем (2,4,8) и 840 параметров. Обучения пока нет.


Вход: torch.Size([2, 4, 8])
Выход: torch.Size([2, 4, 8])
Количество параметров: 840


### Остановка A

1. Во вторую ветвь поступает исходный x или результат первого residual-сложения?
2. Почему norm_1 и norm_2 создаём как два разных модуля?
3. Получили ли мы уже вероятности следующих токенов, вызвав block(x)? Что находится в y?

**Мои ответы:**

1. У нас в коде поступает от преобразованного после attention
2. Потому что это разные слои
3. нет, мы получили score выход после нашего блока

Сохрани код, outputs и ответы, пришли на разбор. B — после A.

**Разбор ментора (исходные ответы сохранены):**

Порядок модулей правильный, но в forward нужно заменить оба x += ... на x = x + ... . Для Tensor += меняет сам вход на месте: внешний x тоже меняется, а backward может столкнуться с изменением сохранённого тензора. Обычное присваивание создаёт результат сложения и привязывает локальное имя x к нему.

1. Верно по смыслу: во вторую ветвь поступает результат первого residual-сложения, то есть исходный вход плюс выход attention(norm_1(вход)), а не только выход attention.
2. Нужны отдельные обучаемые gamma и beta: нормализуем представления на двух разных этапах. Один объект LayerNorm, вызванный дважды, разделял бы параметры; это другая архитектура.
3. Верно, что не вероятности. Но y — не scores, а новые векторы признаков токенов формы (B,T,D). Attention scores — отдельный промежуточный тензор внутри attention. Logits по словарю появятся после будущей выходной проекции.


## B. Проследить промежуточные результаты и причинность

Готовый опыт: разложим вызов блока на знакомые операции. Затем изменим последний токен и сравним более ранние выходы. Dropout в этой реализации отсутствует.

`no_grad` отключает построение графа для диагностических вычислений, не меняет веса. Не нужно добавлять его внутрь обучаемого forward.

In [5]:
with torch.no_grad():
    attention_update = block.attention(block.norm_1(x))
    after_attention = x + attention_update
    mlp_update = block.mlp(block.norm_2(after_attention))
    manual_output = after_attention + mlp_update
    direct_output = block(x)

    print('Ветвь attention:', attention_update.shape)
    print('Первое сложение:', after_attention.shape)
    print('Ветвь MLP:', mlp_update.shape)
    print('Ручная сборка совпадает с forward:', torch.allclose(manual_output, direct_output))

    changed_x = x.clone()
    changed_x[:, -1, 0] += 10  # Меняем один признак последнего токена.
    changed_output = block(changed_x)
    print('Ранние позиции не изменились:',
          torch.allclose(direct_output[:, :-1], changed_output[:, :-1]))
    print('Максимальное изменение последней позиции:',
          (direct_output[:, -1] - changed_output[:, -1]).abs().max().item())


Ветвь attention: torch.Size([2, 4, 8])
Первое сложение: torch.Size([2, 4, 8])
Ветвь MLP: torch.Size([2, 4, 8])
Ручная сборка совпадает с forward: True
Ранние позиции не изменились: True
Максимальное изменение последней позиции: 10.226458549499512


### Остановка B

1. Почему MLP и LayerNorm не нарушают причинность после attention?
2. Обновились ли веса от двух вызовов block с разными входами? Почему?

**Мои ответы:**

1. не понял вопроса
2. нет, у нас tourch.no_grad

Дальше подключим embeddings и проекцию на словарь: получим маленькую decoder-only языковую модель. Текущий блок ещё не обучен.

**Разбор ментора (исходные ответы сохранены):**

1. Причинность означает: выход ранней позиции не зависит от будущих входных токенов. LayerNorm(D) использует признаки только текущего токена, MLP тоже работает отдельно с каждой позицией. Поэтому они не добавляют доступ к будущему после causal attention.
2. Веса не обновились, но причина — отсутствие optimizer.step() или ручной записи в параметры. no_grad отключает построение графа, а не запрещает изменение тензоров. В данном forward вход x как раз изменяется через += даже внутри no_grad.

Сохранены два False. Второй опыт загрязнён изменением x внутри block(x): changed_x клонируется уже после этого изменения, и сравниваются не два входа, отличающихся только последним токеном. Первый False нельзя уверенно объяснить только += по показанному коду: нужны свежие outputs после исправления и последовательного выполнения. Заменить обе строки forward на обычное сложение, затем самостоятельно Restart Kernel → Run All → сохранить. Ментор код и проверки не запускал; текущую реализацию завершённой пока не считаем.


## Повторный разбор после исправления

В forward оба += заменены на x = x + ... . По новым сохранённым outputs ручная сборка совпала с forward (True), более ранние позиции при изменении последнего токена сохранились (True). Форма выхода (2,4,8), параметров 840. Предыдущее замечание о двух False относится к старому выполнению; исправление принято. Урок сборки блока завершён. Это проверка forward по сохранённым результатам, не проверка backward и не обучение модели. Ментор код не запускал.
